In [1]:
import pandas as pd

orders = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
order_payments = pd.read_csv('olist_order_payments_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

# Quick check
print(orders.shape)
print(orders.head())

(99441, 8)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00       

In [2]:
# 1. Orders + Customers (to get customer state)
df = orders.merge(customers[['customer_id', 'customer_state', 'customer_city']], 
                  on='customer_id', how='left')

# 2. + Order items (to get product and seller info)
df = df.merge(order_items[['order_id', 'product_id', 'seller_id', 'price', 'freight_value']], 
              on='order_id', how='left')

# 3. + Products (to get category)
df = df.merge(products[['product_id', 'product_category_name']], 
              on='product_id', how='left')

# 4. + Category translation (Portuguese → English)
df = df.merge(category_translation, 
              on='product_category_name', how='left')

# 5. + Sellers (to get seller state)
df = df.merge(sellers[['seller_id', 'seller_state', 'seller_city']], 
              on='seller_id', how='left')

# 6. + Reviews (to get review score)
reviews_deduped = order_reviews.groupby('order_id', as_index=False)['review_score'].mean()
df = df.merge(reviews_deduped[['order_id', 'review_score']], 
              on='order_id', how='left')

print(df.shape)
print(df.columns.tolist())

(113425, 19)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_state', 'customer_city', 'product_id', 'seller_id', 'price', 'freight_value', 'product_category_name', 'product_category_name_english', 'seller_state', 'seller_city', 'review_score']


In [3]:
# See null counts per column
null_summary = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(pd.DataFrame({'null_count': null_summary, 'null_%': null_pct}))

                               null_count  null_%
order_id                                0    0.00
customer_id                             0    0.00
order_status                            0    0.00
order_purchase_timestamp                0    0.00
order_approved_at                     161    0.14
order_delivered_carrier_date         1968    1.74
order_delivered_customer_date        3229    2.85
order_estimated_delivery_date           0    0.00
customer_state                          0    0.00
customer_city                           0    0.00
product_id                            775    0.68
seller_id                             775    0.68
price                                 775    0.68
freight_value                         775    0.68
product_category_name                2378    2.10
product_category_name_english        2402    2.12
seller_state                          775    0.68
seller_city                           775    0.68
review_score                          961    0.85


In [4]:
# 4b - Fix date columns
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 4c - Handle nulls
df = df.dropna(subset=['order_delivered_customer_date', 
                        'order_estimated_delivery_date'])
df['review_score'] = df['review_score'].fillna(df['review_score'].median())
df['product_category_name_english'] = \
    df['product_category_name_english'].fillna('unknown')

# 4d - Fix data types
df['review_score'] = df['review_score'].astype(float)
df['price'] = df['price'].astype(float)
df['freight_value'] = df['freight_value'].astype(float)

# 4e - Keep only delivered orders
print(df['order_status'].value_counts())
df = df[df['order_status'] == 'delivered']

print(f"\nRows remaining: {len(df)}")

order_status
delivered    110189
canceled          7
Name: count, dtype: int64

Rows remaining: 110189


In [5]:
# 1. Delivery delay in days (positive = late, negative = early)
df['delivery_delay_days'] = (
    df['order_delivered_customer_date'] - 
    df['order_estimated_delivery_date']
).dt.days

# 2. On-time flag (1 = late, 0 = on time or early)
df['is_late'] = (df['delivery_delay_days'] > 0).astype(int)

# 3. Actual delivery time in days
df['actual_delivery_days'] = (
    df['order_delivered_customer_date'] - 
    df['order_purchase_timestamp']
).dt.days

# 4. Order month and year
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['order_month'] = df['order_purchase_timestamp'].dt.month
df['order_year_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Sanity check
print(df[['delivery_delay_days', 'is_late', 'actual_delivery_days']].describe())

on_time_rate = (df['is_late'] == 0).mean() * 100
print(f"\nOn-time delivery rate: {on_time_rate:.1f}%")

       delivery_delay_days        is_late  actual_delivery_days
count        110189.000000  110189.000000         110189.000000
mean            -12.029041       0.065923             12.007342
std              10.158194       0.248149              9.451153
min            -147.000000       0.000000              0.000000
25%             -17.000000       0.000000              6.000000
50%             -13.000000       0.000000             10.000000
75%              -7.000000       0.000000             15.000000
max             188.000000       1.000000            209.000000

On-time delivery rate: 93.4%


In [6]:
df.to_csv('olist_cleaned.csv', index=False)
print(f"Exported {len(df)} rows and {len(df.columns)} columns")
print("Columns:", df.columns.tolist())

Exported 110189 rows and 25 columns
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_state', 'customer_city', 'product_id', 'seller_id', 'price', 'freight_value', 'product_category_name', 'product_category_name_english', 'seller_state', 'seller_city', 'review_score', 'delivery_delay_days', 'is_late', 'actual_delivery_days', 'order_year', 'order_month', 'order_year_month']


In [18]:
# Calculate average estimated delivery time
df['estimated_delivery_days'] = (
    df['order_estimated_delivery_date'] - 
    df['order_purchase_timestamp']
).dt.days

print(f"Avg estimated delivery days: {df['estimated_delivery_days'].mean():.1f}")
print(f"Avg actual delivery days: {df['actual_delivery_days'].mean():.1f}")
print(f"Avg delay days: {df['delivery_delay_days'].mean():.1f}")

Avg estimated delivery days: 23.4
Avg actual delivery days: 12.0
Avg delay days: -12.0
